#### Translating Procedures to WASM

This exercise comes with a modified Python P0 library: for testing purposes, the library function `read()` always returns `41`.

In [ ]:
def runpywasm(wasmfile):
  from pywasm.core import Machine, Runtime, FuncType, ValType

  # P0lib implementation in Python
  def write(_: Machine, args: list[int]) -> list[int]:
    print(args[0])
    return []

  def writeln(_: Machine, args: list[int]) -> list[int]:
    print()
    return []

  def read(_: Machine, args: list[int]) -> list[int]:
    return [41]  # [int(input())]

  # Create runtime
  runtime = Runtime()
  runtime.imports = {
    'P0lib': {
      'write': runtime.allocate_func_host(FuncType([ValType.i32()], []), write),
      'writeln': runtime.allocate_func_host(FuncType([], []), writeln),
      'read': runtime.allocate_func_host(FuncType([], [ValType.i32()]), read),
    }
  }
  # Create and run instance
  instance = runtime.instance_from_file(wasmfile)

The task is to implement the following program in WebAssembly. The first procedure is a linear congruential random number generator with a seed value read when the program starts. The second procedure is Euclid's algorithm for the greatest common divisor. Convert the textual form to the binary form using `wat2wasm` and test your code.

```Pascal
var r: integer

procedure randint(bound: integer) → (rand: integer)
  const a = 16807
  const c = 11
  const m = 65535
    r := (a × r + c) mod m
    rand := r mod bound

procedure gcd(x: integer; y: integer) → (d: integer)
  while x ≠ y do
    if x > y then x := x - y
    else y := y - x
  d := x

program randgcd
  var x, y, d: integer
    r ← read()
    x ← randint(100); write(x)
    y ← randint(100); write(y)
    d ← gcd(x, y); write(d)
```

<div style="float:right;background-color:lightgrey;border-left:20px solid white">

<pre>
;;  program setx
(func $program
  ;;  var x: integer
  (local $x i32)
  ;;    x := 0
  i32.const 0
  local.set $x
)
</pre>

</div>

<div style="float:right;background-color:lightgrey;border-left:20px solid white">

```Pascal
program setx
  var x: integer
    x := 0
```

</div>

Your WebAssembly code must follow the translation scheme from the course notes. In particular, it must call `read` and `write` as specified above. Every section of your WebAssembly code must be preceded by the corresponding P0 fragment, as illustrated to the right.

In [ ]:
%%writefile randgcd.wat
(module
(import "P0lib" "write" (func $write (param i32)))
(import "P0lib" "writeln" (func $writeln))
(import "P0lib" "read" (func $read (result i32)))
;;  var r: integer
(global $r (mut i32) i32.const 0)
;;  procedure randint(bound: integer) → (rand: integer)
(func $randint (param $bound i32) (result i32)
(local $rand i32)
(local $0 i32)
  ;;  const a = 16807
  ;;  const c = 11
  ;;  const m = 65535
  ;;    r := (a × r + c) mod m
  i32.const 16807
  global.get $r
  i32.mul
  i32.const 11
  i32.add
  i32.const 65535
  i32.rem_s
  global.set $r
  ;;    rand := r mod bound
  global.get $r
  local.get $bound
  i32.rem_s
  local.set $rand
  local.get $rand
)
;;  procedure gcd(x: integer, y: integer) → (d: integer)
(func $gcd (param $x i32) (param $y i32) (result i32)
(local $d i32)
(local $0 i32)
  ;;  while x ≠ y do
  loop
    local.get $x
    local.get $y
    i32.ne
    if
      ;;    if x > y then x := x - y
      local.get $x
      local.get $y
      i32.gt_s
      if
        ;;      x := x - y
        local.get $x
        local.get $y
        i32.sub
        local.set $x
      ;;    else y := y - x
      else
        local.get $y
        local.get $x
        i32.sub
        local.set $y
      end
      br 1
    end
  end
  ;;  d := x
  local.get $x
  local.set $d
  local.get $d
)
;;  program randgcd
(func $program
  ;;  var x, y, d: integer
  (local $x i32)
  (local $y i32)
  (local $d i32)
  (local $0 i32)
  ;;    r ← read()
  call $read
  global.set $r
  ;;    x ← randint(100); write(x)
  i32.const 100
  call $randint
  local.set $x
  local.get $x
  call $write
  ;;    y ← randint(100); write(y)
  i32.const 100
  call $randint
  local.set $y
  local.get $y
  call $write
  ;;    d ← gcd(x, y); write(d)
  local.get $x
  local.get $y
  call $gcd
  local.set $d
  local.get $d
  call $write
)
(memory 1)
(start $program)
)

In [ ]:
!wat2wasm randgcd.wat

In [ ]:
runpywasm('randgcd.wasm')

You may test your program with:

In [ ]:
%%capture output
runpywasm('randgcd.wasm')

In [ ]:
assert str(output) == '48\n57\n3\n'

The cells below are for grading purposes only; please ignore them.

In [ ]:
%%capture output